# YOLOX → ONNX Export

학습된 `best_ckpt.pth`를 ONNX 포맷으로 변환해 Google Drive에 저장한다.

**실행 순서:** 셀을 위에서 아래로 순서대로 실행하면 된다.

## 1. YOLOX 설치

In [ ]:
import os
from pathlib import Path

YOLOX_HOME = Path("/content/YOLOX")

if not YOLOX_HOME.exists():
    !git clone https://github.com/Megvii-BaseDetection/YOLOX.git {str(YOLOX_HOME)}

os.chdir(YOLOX_HOME)

!pip install -q numpy loguru tqdm tabulate thop ninja psutil
!pip install -q opencv-python-headless "pycocotools>=2.0.2"
!pip install -q onnx onnxsim onnxruntime
!pip install -q -e . --no-deps

print("설치 완료")

## 2. Google Drive 마운트

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

# best_ckpt.pth 경로 — 기존 학습 노트북에서 저장된 위치
BEST_CKPT = Path("/content/drive/MyDrive/계량기프로젝트/best_ckpt.pth")
# ONNX 파일도 같은 Drive 폴더에 저장
ONNX_OUT  = Path("/content/drive/MyDrive/계량기프로젝트/meter_yolox_s.onnx")

print("best_ckpt.pth 존재:", BEST_CKPT.exists())

## 3. export_onnx.py 패치

YOLOX의 `export_onnx.py`는 `torch.load`에 `weights_only` 인자가 없어서
PyTorch 2.6+에서 numpy scalar 로드 실패 에러가 난다.
한 줄 교체로 해결한다.

In [ ]:
export_py = YOLOX_HOME / "tools" / "export_onnx.py"
src = export_py.read_text()

# PyTorch 2.6+: torch.load weights_only 기본값이 True로 바뀌어 numpy scalar 로드 실패
patched = src.replace(
    'ckpt = torch.load(ckpt_file, map_location="cpu")',
    'ckpt = torch.load(ckpt_file, map_location="cpu", weights_only=False)'
)

# PyTorch 최신 버전: torch.onnx._export가 제거되고 torch.onnx.export로 변경됨
patched = patched.replace(
    'torch.onnx._export(',
    'torch.onnx.export('
)

export_py.write_text(patched)
print("weights_only 패치:", 'weights_only=False' in patched)
print("_export 패치:", 'torch.onnx.export(' in patched)

## 4. Exp 파일 생성

학습 때 사용한 하이퍼파라미터를 그대로 재현해야 모델 구조가 일치한다.

In [ ]:
exp_content = '''#!/usr/bin/env python3
from yolox.exp import Exp as MyExp

class Exp(MyExp):
    def __init__(self):
        super().__init__()
        self.depth = 0.33
        self.width = 0.50
        self.num_classes = 1
        self.input_size = (640, 640)
        self.test_size = (640, 640)
        self.exp_name = "meter_yolox_s"
'''

EXP_FILE = YOLOX_HOME / "exps" / "meter_yolox_s.py"
EXP_FILE.write_text(exp_content)
print("Exp 파일 작성:", EXP_FILE)

## 5. ONNX Export 실행

In [ ]:
!python tools/export_onnx.py \
    -f exps/meter_yolox_s.py \
    -c {str(BEST_CKPT)} \
    --output-name {str(ONNX_OUT)} \
    -o 12

if ONNX_OUT.exists():
    print(f"\nONNX 저장 완료: {ONNX_OUT}  ({ONNX_OUT.stat().st_size / 1e6:.1f} MB)")
else:
    print("\nONNX export 실패 — 위 로그를 확인하세요.")